# Period moves — outrights + curves/flies/spreads

**Fuente de datos**
- `SOURCE = "csv"` → `data_11_22.csv` (dump manual; recomendado ahora)
- `SOURCE = "live"` → `sofr_live.csv` (BBG; `REFRESH=False` si refdata está caído)

Edita **START** / **END** y Run All.


In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "lib" / "period_moves.py").exists() else _DZ
if not (REPO / "lib" / "period_moves.py").exists():
    raise FileNotFoundError("No lib/period_moves.py — git pull origin main")

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())


In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Inputs ───────────────────────────────────────────────────────────────────
START = "10/15/26 09:00"
END   = "10/16/26 12:00"   # o "" → asof del panel

SOURCE = "csv"             # "csv" | "live"
CSV_PATH = "data_11_22.csv"
REFRESH = False            # solo aplica a SOURCE="live"
DAILY = False

import importlib
from pathlib import Path
import pandas as pd
import curve_config
import lib.data as data
import lib.period_moves as period_moves

importlib.reload(curve_config)
importlib.reload(data)
importlib.reload(period_moves)

outrights_cfg = list(getattr(curve_config, "MOVE_OUTRIGHTS", ["2", "5", "10", "30"]))
structures_cfg = list(getattr(
    curve_config, "MOVE_STRUCTURES",
    ["2s5s", "5s10s", "10s30s", "2s5s10s", "5s10s30s", "USSFCT05", "USSFCT10", "USSFCT30"],
))

end_arg = END.strip() if isinstance(END, str) and END.strip() else pd.Timestamp.now().strftime("%m/%d/%y %H:%M")
print("SOURCE:", SOURCE, " CSV:", CSV_PATH if SOURCE == "csv" else "—")
print("START:", period_moves.parse_ts(START), " END:", period_moves.parse_ts(end_arg))
if SOURCE == "csv":
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError("Missing %s — put the dump in the repo root" % CSV_PATH)
else:
    store = data.resolve_live_store_path()
    print("store:", store, "exists=", store.exists())



In [ ]:
kw = dict(
    outrights=outrights_cfg,
    structures=structures_cfg,
    refresh=REFRESH if SOURCE == "live" else False,
    prefer_daily=DAILY,
)
if SOURCE == "csv":
    kw["csv_path"] = CSV_PATH

outrights, structures, meta, panel = period_moves.run_period_moves(START, end_arg, **kw)
print("horizon:", meta["horizon"], "| panel asof:", panel.index[-1], "| n=", len(panel))
print("OK — tablas abajo")



In [ ]:
# ── Tabla 1: OUTRIGHTS (2y/5y/10y/30y) en bp ─────────────────────────────────
print("OUTRIGHTS — change & std in bp  |  z = change / std(same-horizon moves)")
display(outrights)

In [ ]:
# ── Tabla 2: CURVES / FLIES / SPREADS (bp / BBG units) ───────────────────────
print("STRUCTURES — 2s5s, 5s10s, 10s30s, flies, USSFCT…")
display(structures)

In [ ]:
# Guardar CSV (opcional)
out_dir = Path("graphs/period_moves")
out_dir.mkdir(parents=True, exist_ok=True)
outrights.to_csv(out_dir / "moves_outrights.csv", index=False)
structures.to_csv(out_dir / "moves_structures.csv", index=False)
print("wrote", out_dir.resolve())